In [14]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [15]:
df = pd.read_csv("news_categories.csv")
df.head()

,category,news,summary
0,business,Ad sales boost Time Warner profit\n\nQuarterly...,TimeWarner said fourth quarter sales rose 2% t...
1,business,Dollar gains on Greenspan speech\n\nThe dollar...,The dollar has hit its highest level against t...
2,business,Yukos unit buyer faces loan claim\n\nThe owner...,Yukos' owner Menatep Group says it will ask Ro...
3,business,High fuel prices hit BA's profits\n\nBritish A...,"Rod Eddington, BA's chief executive, said the ..."
4,business,Pernod takeover talk lifts Domecq\n\nShares in...,Pernod has reduced the debt it took on to fund...


In [16]:
df.info()
df=df.drop(["news"],axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  2225 non-null   object
 1   news      2225 non-null   object
 2   summary   2225 non-null   object
dtypes: object(3)
memory usage: 52.3+ KB


In [17]:
#Encoding labels
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["category"] = encoder.fit_transform(df["category"])
df.head()

,category,summary
0,0,TimeWarner said fourth quarter sales rose 2% t...
1,0,The dollar has hit its highest level against t...
2,0,Yukos' owner Menatep Group says it will ask Ro...
3,0,"Rod Eddington, BA's chief executive, said the ..."
4,0,Pernod has reduced the debt it took on to fund...


In [18]:
#Dropped news
df = df.rename(columns={"summary":"news"})
df.head()

,category,news
0,0,TimeWarner said fourth quarter sales rose 2% t...
1,0,The dollar has hit its highest level against t...
2,0,Yukos' owner Menatep Group says it will ask Ro...
3,0,"Rod Eddington, BA's chief executive, said the ..."
4,0,Pernod has reduced the debt it took on to fund...


In [46]:
df["news"] = df["news"].fillna("").astype(str)
X=df["news"]

news = X.tolist()
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(news)

word_index = tokenizer.word_index
vocab_size = len(word_index)
sequences = tokenizer.texts_to_sequences(news)
X = pad_sequences(sequences,maxlen=500,padding='post',truncating='post')

y=df["category"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # Proportion of the dataset to use for testing (e.g., 20%)
    random_state=42,    # Ensures reproducibility
    stratify=y          # Ensures equal/proportional split based on labels
)

In [47]:
#Embedding
embedding_index={}
embedding_dim = 50
with open ("Downloads/glove.6B/glove.6B.50d.txt",'r',encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:],dtype='float32')
        embedding_index[word] = coefs
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word,i in word_index.items():
        if i<vocab_size:
            embedding_vector=embedding_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i]=embedding_vector

In [50]:
num_classes = len(np.unique(y_train))
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(128, return_sequences=True)
    ),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)              │ ?                           │       1,091,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d (SpatialDropout1D) │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_6 (Conv1D)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ ?                           │     0 (unbuilt) │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_6 (MaxPooling1D)       │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,091,600 (4.16 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,091,600 (4.16 MB)

In [51]:
history = model.fit(
    np.array(X_train), 
    np.array(y_train), 
    epochs=50, 
    validation_data=(np.array(X_test), np.array(y_test)), 
    verbose=2
)

Epoch 1/50
56/56 - 16s - 294ms/step - accuracy: 0.5612 - loss: 1.1833 - val_accuracy: 0.8854 - val_loss: 0.9721
Epoch 2/50
56/56 - 13s - 241ms/step - accuracy: 0.8635 - loss: 0.4643 - val_accuracy: 0.8966 - val_loss: 0.4208
Epoch 3/50
56/56 - 13s - 230ms/step - accuracy: 0.8994 - loss: 0.3105 - val_accuracy: 0.9438 - val_loss: 0.3009
Epoch 4/50
56/56 - 13s - 229ms/step - accuracy: 0.9135 - loss: 0.2667 - val_accuracy: 0.9461 - val_loss: 0.1875
Epoch 5/50
56/56 - 13s - 225ms/step - accuracy: 0.9180 - loss: 0.2383 - val_accuracy: 0.9528 - val_loss: 0.1475
Epoch 6/50
56/56 - 13s - 234ms/step - accuracy: 0.9337 - loss: 0.2099 - val_accuracy: 0.9640 - val_loss: 0.1234
Epoch 7/50
56/56 - 13s - 228ms/step - accuracy: 0.9371 - loss: 0.1810 - val_accuracy: 0.9640 - val_loss: 0.1218
Epoch 8/50
56/56 - 13s - 233ms/step - accuracy: 0.9399 - loss: 0.1692 - val_accuracy: 0.9528 - val_loss: 0.1282
Epoch 9/50
56/56 - 13s - 229ms/step - accuracy: 0.9438 - loss: 0.1736 - val_accuracy: 0.9663 - val_loss:

In [53]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

print(classification_report(
    np.array(y_test),
    predicted_classes,
    target_names=encoder.classes_
))

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step
               precision    recall  f1-score   support

     business       0.98      0.93      0.95       102
entertainment       0.95      0.99      0.97        77
     politics       0.96      0.95      0.96        84
        sport       1.00      1.00      1.00       102
         tech       0.95      0.99      0.97        80

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted avg       0.97      0.97      0.97       445



In [55]:
loss, accuracy = model.evaluate(
    X_test,
    np.array(y_test),
    verbose=0
)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.9707865118980408


In [56]:
model.save("news_category_model.keras")